<a href="https://colab.research.google.com/github/chaunijs/onlineshoppingprice/blob/main/7_eleven_allonline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb
!pip install selenium webdriver-manager beautifulsoup4 pandas xlsxwriter

In [2]:
from google.colab import data_table
data_table.enable_dataframe_formatter()

In [3]:
# from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import polars as pl
import asyncio
# import xlsxwriter
import datetime
from datetime import date
import IPython.display as display
import datetime
import xlsxwriter
today_date = datetime.datetime.now().strftime("%Y-%m-%d")
print("Today is",today_date)

Today is 2026-04-20


In [7]:
# @title scrape_7eleven_data
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import polars as pl

def scrape_7eleven_data(base_url: str) -> pl.DataFrame:
    """
    Scrapes product data from 7-Eleven's online store, handling pagination,
    and returns a cleaned Polars DataFrame.

    Args:
        base_url (str): The base URL for the product category, without the "p=" parameter.

    Returns:
        pl.DataFrame: A Polars DataFrame containing the scraped product data.
    """
    # 1. Setup Chrome options
    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

    print("Starting browser...")

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)

    data = []
    p_index = 0
    previous_page_names = []

    while True:
        print(f"\nScraping Page {p_index + 1} (URL parameter p={p_index})...")

        current_page_url = base_url + f"&p={p_index}"
        driver.get(current_page_url)

        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CLASS_NAME, "item-description-cls-mobile"))
            )
        except Exception as e:
            print("No products found on this page. Reached the end!")
            break

        soup = BeautifulSoup(driver.page_source, "html.parser")
        product_titles = soup.find_all("div", class_="item-description-cls-mobile")

        if not product_titles:
            print("Page is empty. Reached the end!")
            break

        current_page_names = [title.text.strip() for title in product_titles]

        if current_page_names == previous_page_names:
            print("Duplicate items detected! We have reached the final page.")
            break

        previous_page_names = current_page_names.copy()

        # Parse Data
        for title_elem in product_titles:
            name = title_elem.text.strip()
            parent = title_elem.parent
            promotion_price_elem = None

            levels_climbed = 0
            while parent and parent.name != 'body' and levels_climbed < 5:
                promotion_price_elem = parent.find("strong")
                if promotion_price_elem:
                    break
                parent = parent.parent
                levels_climbed += 1

            promotion_price = promotion_price_elem.text.strip() if promotion_price_elem else None
            original_price_elem = parent.find("s") if parent else None
            original_price = original_price_elem.text.strip() if original_price_elem else None

            data.append({
                "name": name,
                "promotion_price": promotion_price,
                "original_price": original_price
            })

        print(f"Successfully grabbed {len(product_titles)} items from Page {p_index + 1}.")

        p_index += 1

    driver.quit()

    # Clean Data using Polars
    df = pl.DataFrame(data)

    if not df.is_empty():
        df = df.with_columns(
            pl.col("promotion_price").cast(pl.String).str.replace_all(r"[^\d.]", "").cast(pl.Float64, strict=False).alias("promotion_price"),
            pl.col("original_price").cast(pl.String).str.replace_all(r"[^\d.]", "").cast(pl.Float64, strict=False).alias("original_price")
        )

        # Drop duplicates
        df = df.unique(subset=['name'], keep='first')

        print("\n--- Final Scraping Results ---")
        print(df.tail(10))
        print(f"\nFinal clean item count: {df.height}")

        return df
    else:
        print("No data was scraped.")
        return pl.DataFrame()

# Example usage of the new function with multiple URLs:
# Define a list of URLs to scrape
list_of_7eleven_urls = [
    "https://allonline.7eleven.co.th/supermarket/household-items/laundry/liquid-detergent/?filter.BRAND=Fineline&filter.BRAND=Hygiene&filter.BRAND=%E0%B9%80%E0%B8%9B%E0%B8%B2&filter.BRAND=%E0%B9%81%E0%B8%AD%E0%B8%97%E0%B9%81%E0%B8%97%E0%B8%84&filter.PRICE=49-470&filter.initial_from_PRICE=49&filter.initial_to_PRICE=470&landing=true&pageSize=90&sortBy=si&view=0",
    # Add more URLs here if needed, for example:
    "https://allonline.7eleven.co.th/supermarket/household-items/dish-detergent/?show=all&filter.from_PRICE=53&filter.to_PRICE=899&brands=on&sortBy=si&filter.BRAND=%E0%B9%84%E0%B8%A5%E0%B8%9B%E0%B8%AD%E0%B8%99%E0%B9%80%E0%B8%AD%E0%B8%9F&filter.initial_from_PRICE=53&filter.initial_to_PRICE=899&view=6",
]

all_scraped_dataframes = []

for url in list_of_7eleven_urls:
    print(f"\n--- Starting scraping for URL: {url} ---")
    df_current_url = scrape_7eleven_data(url)
    if not df_current_url.is_empty():
        all_scraped_dataframes.append(df_current_url)

# Concatenate all dataframes if any data was scraped
if all_scraped_dataframes:
    df_combined_7eleven = pl.concat(all_scraped_dataframes, how='vertical')
    print(f"\nCombined data from all URLs. Total items before final duplicate check: {df_combined_7eleven.height}")

    # Drop duplicates from the combined DataFrame based on 'name'
    df_combined_7eleven = df_combined_7eleven.unique(subset=['name'], keep='first')
    print(f"Combined data after dropping duplicates. Final unique item count: {df_combined_7eleven.height}")

    filename = 'combined_scoped_detergents.csv'
    df_combined_7eleven.write_csv(filename, separator=',', include_header=True)
    print(f"\nCombined data successfully saved to {filename}!")
else:
    print("No data was scraped from any of the provided URLs.")



--- Starting scraping for URL: https://allonline.7eleven.co.th/supermarket/household-items/laundry/liquid-detergent/?filter.BRAND=Fineline&filter.BRAND=Hygiene&filter.BRAND=%E0%B9%80%E0%B8%9B%E0%B8%B2&filter.BRAND=%E0%B9%81%E0%B8%AD%E0%B8%97%E0%B9%81%E0%B8%97%E0%B8%84&filter.PRICE=49-470&filter.initial_from_PRICE=49&filter.initial_to_PRICE=470&landing=true&pageSize=90&sortBy=si&view=0 ---
Starting browser...

Scraping Page 1 (URL parameter p=0)...
Successfully grabbed 90 items from Page 1.

Scraping Page 2 (URL parameter p=1)...
Successfully grabbed 11 items from Page 2.

Scraping Page 3 (URL parameter p=2)...
Duplicate items detected! We have reached the final page.

--- Final Scraping Results ---
shape: (10, 3)
┌─────────────────────────────┬─────────────────┬────────────────┐
│ name                        ┆ promotion_price ┆ original_price │
│ ---                         ┆ ---             ┆ ---            │
│ str                         ┆ f64             ┆ f64            │
╞═══════

In [5]:
# @title Scrape Specific WatchList Product URL
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import polars as pl

def get_driver():
    """Helper to initialize the Selenium driver[cite: 20]."""
    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    # Using a modern user agent [cite: 4]
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=chrome_options)

def scrape_specific_products(urls: list) -> pl.DataFrame:
    """
    Scrapes data from a list of specific 7-Eleven product detail pages[cite: 19].
    """
    driver = get_driver()
    data = []

    for url in urls:
        print(f"Scraping product: {url}")
        try:
            driver.get(url)

            # Wait for a more generic element to ensure the page has loaded at all
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.TAG_NAME, "h1"))
            )

            soup = BeautifulSoup(driver.page_source, "html.parser")

            # 1. Extract Product Name
            name_elem = soup.find("h1")
            name = name_elem.text.strip() if name_elem else "Unknown Name"

            # 2. Extract Prices with multiple fallback selectors
            # Logic: Try different common classes for AllOnline detail pages
            promo_elem = soup.select_one(".current-price, .price-promotion, .price-now")
            original_elem = soup.select_one(".price-strike, .price-old, s, del")

            promo_text = promo_elem.text.strip() if promo_elem else None
            orig_text = original_elem.text.strip() if original_elem else None

            data.append({
                "name": name,
                "promotion_price": promo_text,
                "original_price": orig_text,
                "url": url
            })

            print(f"  Successfully grabbed: {name} (Price: {promo_text})")

        except Exception as e:
            print(f"  Error scraping {url}: {str(e)[:100]}...") # Print a truncated error

        time.sleep(2) # Increased delay to ensure stability

    driver.quit()

    # Clean Data using Polars
    df = pl.DataFrame(data)
    if not df.is_empty():
        df = df.with_columns(
            pl.col("promotion_price").cast(pl.String).str.replace_all(r"[^\d.]", "").cast(pl.Float64, strict=False),
            pl.col("original_price").cast(pl.String).str.replace_all(r"[^\d.]", "").cast(pl.Float64, strict=False)
        )
        # Added URL tracking column [cite: 24]
    return df

# --- Execution ---
specific_product_urls = [
    "https://allonline.7eleven.co.th/p/%E0%B9%84%E0%B8%9F%E0%B8%99%E0%B9%8C%E0%B9%84%E0%B8%A5%E0%B8%99%E0%B9%8C-%E0%B8%9E%E0%B8%A5%E0%B8%B1%E0%B8%AA-%E0%B8%99%E0%B9%89%E0%B8%B3%E0%B8%A2%E0%B8%B2%E0%B8%8B%E0%B8%B1%E0%B8%81%E0%B8%9C%E0%B9%89%E0%B8%B2-%E0%B8%8B%E0%B8%B1%E0%B8%99%E0%B8%99%E0%B8%B5%E0%B9%88-%E0%B9%82%E0%B8%81%E0%B8%A5%E0%B8%94%E0%B9%8C-1250-%E0%B8%A1%E0%B8%A5/346148/",
    "https://allonline.7eleven.co.th/p/%E0%B9%80%E0%B8%9B%E0%B8%B2%E0%B8%A7%E0%B8%B4%E0%B8%99%E0%B8%A7%E0%B8%AD%E0%B8%8A-%E0%B8%99%E0%B9%89%E0%B8%B3%E0%B8%A2%E0%B8%B2%E0%B8%8B%E0%B8%B1%E0%B8%81%E0%B8%9C%E0%B9%89%E0%B8%B2-%E0%B8%A5%E0%B8%B4%E0%B8%84%E0%B8%A7%E0%B8%B4%E0%B8%94-%E0%B8%AA%E0%B8%B5%E0%B8%9F%E0%B9%89%E0%B8%B2-620-%E0%B8%A1%E0%B8%A5/461013/"
]

df_watchlist = scrape_specific_products(specific_product_urls)
print("\n--- Final Results ---")
print(df_watchlist)

Scraping product: https://allonline.7eleven.co.th/p/%E0%B9%84%E0%B8%9F%E0%B8%99%E0%B9%8C%E0%B9%84%E0%B8%A5%E0%B8%99%E0%B9%8C-%E0%B8%9E%E0%B8%A5%E0%B8%B1%E0%B8%AA-%E0%B8%99%E0%B9%89%E0%B8%B3%E0%B8%A2%E0%B8%B2%E0%B8%8B%E0%B8%B1%E0%B8%81%E0%B8%9C%E0%B9%89%E0%B8%B2-%E0%B8%8B%E0%B8%B1%E0%B8%99%E0%B8%99%E0%B8%B5%E0%B9%88-%E0%B9%82%E0%B8%81%E0%B8%A5%E0%B8%94%E0%B9%8C-1250-%E0%B8%A1%E0%B8%A5/346148/
  Successfully grabbed: ไฟน์ไลน์ พลัส น้ำยาซักผ้า ซันนี่ โกลด์ 1250 มล. (Price: None)
Scraping product: https://allonline.7eleven.co.th/p/%E0%B9%80%E0%B8%9B%E0%B8%B2%E0%B8%A7%E0%B8%B4%E0%B8%99%E0%B8%A7%E0%B8%AD%E0%B8%8A-%E0%B8%99%E0%B9%89%E0%B8%B3%E0%B8%A2%E0%B8%B2%E0%B8%8B%E0%B8%B1%E0%B8%81%E0%B8%9C%E0%B9%89%E0%B8%B2-%E0%B8%A5%E0%B8%B4%E0%B8%84%E0%B8%A7%E0%B8%B4%E0%B8%94-%E0%B8%AA%E0%B8%B5%E0%B8%9F%E0%B9%89%E0%B8%B2-620-%E0%B8%A1%E0%B8%A5/461013/
  Successfully grabbed: เปาวินวอช น้ำยาซักผ้า ลิควิด สีฟ้า 620 มล. (Price: None)

--- Final Results ---
shape: (2, 4)
┌───────────────────────────┬──────

In [9]:
df_watchlist.drop_in_place('url')
df_combined_7eleven = pl.concat([df_combined_7eleven, df_watchlist])

In [ ]:
df_combined_7eleven

name,promotion_price,original_price
str,f64,f64
"""ไฮยีน เอ็กซ์เพิร์ท วอช ผลิตภัณ…",114.0,139.0
"""ไฮยีน เอ็กซ์เพิร์ทวอช กลิ่นสปร…",100.0,130.0
"""ไฮยีน น้ำยาซักผ้า พีโอนีบลูม 1…",114.0,139.0
"""ไฮยีน น้ำยาซักผ้า มิราเคิล 600…",100.0,130.0
"""ไฟน์ไลน์ พลัส น้ำยาซักผ้า สำหร…",89.0,null
…,…,…
"""แอทแทค น้ำยาซักผ้ากลิ่นคลีน แอ…",180.0,null
"""เปา วินวอชลิควิด ถุงเติม น้ำยา…",185.0,null
"""ไฮยีน น้ำยาซักผ้า แฮปปี้ซันชาย…",119.0,null


In [10]:
# @title udf data-prep
import polars as pl

def re_evaluate_price(df: pl.DataFrame) -> pl.DataFrame:
    """
    Standardizes pricing logic:
    1. If original_price is missing, move the promotion_price to it.
    2. If promotion_price matches the original_price, set it to Null.
    """
    return (
        df.with_columns(
            # Step 1: Fix missing original_prices by 'swapping' from promotion_price
            pl.when(pl.col("original_price").is_null() & pl.col("promotion_price").is_not_null())
            .then(pl.col("promotion_price"))
            .otherwise(pl.col("original_price"))
            .alias("original_price")
        )
        .with_columns(
            # Step 2: Now that original_price is populated, nullify redundant promotions
            pl.when(pl.col("promotion_price") == pl.col("original_price"))
            .then(None)
            .otherwise(pl.col("promotion_price"))
            .alias("promotion_price")
        )
    )

# Apply the preparation function
df_prep_7_eleven = re_evaluate_price(df_combined_7eleven)

print("Data has been re-evaluated.")


Data has been re-evaluated.


In [11]:
df_prep_7_eleven

name,promotion_price,original_price
str,f64,f64
"""แอทแทค น้ำยาซักผ้า ไลฟ์ลี่ บลู…",null,95.0
"""ไฟน์ไลน์ น้ำยาซักผ้า แฮปปี้เนส…",null,69.0
"""ไฟน์ไลน์ พลัส น้ำยาซักผ้า สีทอ…",89.0,90.0
"""ไฮยีน น้ำยาซักผ้า แฮปปี้ซันชาย…",100.0,130.0
"""ไฮยีน เอ็กซ์เพิร์ท วอช ผลิตภัณ…",100.0,130.0
…,…,…
"""ไฟน์ไลน์ น้ำยาซักผ้า แฮปปี้เนส…",null,271.0
"""น้ำยาซักผ้าแอทแทค คลีน แอดวานซ…",null,95.0
"""ไฟน์ไลน์ น้ำยาซักผ้า สูตรเข้มข…",150.0,195.0


In [16]:
import polars as pl
from datetime import date
import re

# Standardizing column names to match the requested function logic
# Assuming df_prep_7_eleven is the current working Polars dataframe

def parse_product_names_TH(df: pl.DataFrame, shop_name: str) -> pl.DataFrame:
    """
    Pass any supermarket dataframe through this function to standardize the columns.
    Improved regex to better capture Volume and Unit from Thai product names.
    """
    # 1. Setup the dynamic date
    today_date = date.today().strftime("%Y-%m-%d")

    # 2. Updated patterns
    # Removed strict word boundary \b to handle Thai strings better and added optional space/pack info support
    quant_unit_pattern = r"(?i)([\d,.]+)\s*(มล\.|ลิตร|ก\.ก\.|ML|G|KG|L)"
    pack_pattern = r"(?i)(PACK\s*\d*\s*FREE\s*\d+|PACK\s*\d*\s*\+\s*\d+|PACK\s*\d+|TWINPACK|\bX\s*\d+\b|P?\d+\s*\+\s*\d+|\(\d+\+\d+\)|\d+\s*FREE\s*\d+|\bPACK\b|แพ็ก\s*\d+\s*ชิ้น)"

    # Define Thai brand extraction logic
    thai_brands = ["ไฟน์ไลน์", "ไฮยีน", "เปา", "แอทแทค", "ไลปอนเอฟ"]
    brand_pattern = r"^(" + "|".join(re.escape(b) for b in thai_brands) + r")"

    # 3. Apply the Polars transformation
    return df.with_columns(
        # Add condition column
        pl.lit(None).alias("condition"),

        pl.lit(today_date).alias("Date"),

        # Extract Brand: Check for Thai list first, then fallback to split
        pl.col("name").str.extract(brand_pattern, 1)
          .fill_null(pl.col("name").str.split(" ").list.first())
          .alias("Brand"),

        # Fixed Volume Extraction
        pl.col("name")
          .str.extract(quant_unit_pattern, 1)
          .str.replace_all(",", "")
          .cast(pl.Int64, strict=False)
          .alias("Volume"),

        # Extract Unit
        pl.col("name").str.extract(quant_unit_pattern, 2).str.to_uppercase().alias("Unit"),

        # Extract Pack size
        pl.col("name").str.extract(pack_pattern, 1).str.to_uppercase().alias("Pack"),

        # Add the dynamic Shop identifier
        pl.lit(shop_name).alias("Retailer"),
    )

# Apply the transformation
df_trans_7_eleven = parse_product_names_TH(df_prep_7_eleven.unique(), "7-Eleven")

# Display the results with the new column order
(df_trans_7_eleven.select([
    "name",
    "promotion_price",
    "original_price",
    "condition",
    "Date",
    "Brand",
    "Volume",
    "Unit",
    "Pack",
    "Retailer"
    ]).head(15))

name,promotion_price,original_price,condition,Date,Brand,Volume,Unit,Pack,Retailer
str,f64,f64,null,str,str,i64,str,str,str
"""ไฟน์ไลน์ น้ำยาซักผ้า ดีลักซ์ เ…",89.0,90.0,null,"""2026-04-20""","""ไฟน์ไลน์""",30,"""มล.""",null,"""7-Eleven"""
"""ไฮยีน เอ็กซ์เพิร์ทวอช กลิ่นมอร…",100.0,130.0,null,"""2026-04-20""","""ไฮยีน""",600,"""มล.""",null,"""7-Eleven"""
"""เปา วินวอชลิควิด ถุงเติม น้ำยา…",null,185.0,null,"""2026-04-20""","""เปา""",1500,"""มล.""",null,"""7-Eleven"""
"""ไฮยีน ซันไรซ์คิส น้ำยาซักผ้า ส…",100.0,130.0,null,"""2026-04-20""","""ไฮยีน""",600,"""มล.""",null,"""7-Eleven"""
"""ไฟน์ไลน์ น้ำยาซักผ้า สูตรเข้มข…",null,89.0,null,"""2026-04-20""","""ไฟน์ไลน์""",550,"""มล.""",null,"""7-Eleven"""
…,…,…,…,…,…,…,…,…,…
"""ไฟน์ไลน์ น้ำยาซักผ้า แฮปปี้เนส…",169.0,195.0,null,"""2026-04-20""","""ไฟน์ไลน์""",520,"""มล.""","""แพ็ก 3 ชิ้น""","""7-Eleven"""
"""ไลปอนเอฟ น้ำยาล้างจาน สูตรอนาม…",129.0,165.0,null,"""2026-04-20""","""ไลปอนเอฟ""",3200,"""มล.""",null,"""7-Eleven"""
"""แอทแทค น้ำยาซักผ้า ไลฟ์ลี่ บลู…",null,95.0,null,"""2026-04-20""","""แอทแทค""",700,"""มล.""",null,"""7-Eleven"""


In [17]:
df_trans_7_eleven

name,promotion_price,original_price,condition,Date,Brand,Volume,Unit,Pack,Retailer
str,f64,f64,null,str,str,i64,str,str,str
"""ไฟน์ไลน์ น้ำยาซักผ้า ดีลักซ์ เ…",89.0,90.0,null,"""2026-04-20""","""ไฟน์ไลน์""",30,"""มล.""",null,"""7-Eleven"""
"""ไฮยีน เอ็กซ์เพิร์ทวอช กลิ่นมอร…",100.0,130.0,null,"""2026-04-20""","""ไฮยีน""",600,"""มล.""",null,"""7-Eleven"""
"""เปา วินวอชลิควิด ถุงเติม น้ำยา…",null,185.0,null,"""2026-04-20""","""เปา""",1500,"""มล.""",null,"""7-Eleven"""
"""ไฮยีน ซันไรซ์คิส น้ำยาซักผ้า ส…",100.0,130.0,null,"""2026-04-20""","""ไฮยีน""",600,"""มล.""",null,"""7-Eleven"""
"""ไฟน์ไลน์ น้ำยาซักผ้า สูตรเข้มข…",null,89.0,null,"""2026-04-20""","""ไฟน์ไลน์""",550,"""มล.""",null,"""7-Eleven"""
…,…,…,…,…,…,…,…,…,…
"""แอทแทค น้ำยาซักผ้ากลิ่นคลีน แอ…",219.0,240.0,null,"""2026-04-20""","""แอทแทค""",2100,"""มล.""",null,"""7-Eleven"""
"""ไฟน์ไลน์ น้ำยาซักผ้า โปรคลีน ช…",150.0,195.0,null,"""2026-04-20""","""ไฟน์ไลน์""",550,"""มล.""","""แพ็ก 3 ชิ้น""","""7-Eleven"""
"""ไฟน์ไลน์ผลิตภัณ์ซักผ้าสูตรเข้ม…",150.0,195.0,null,"""2026-04-20""","""ไฟน์ไลน์""",500,"""มล.""","""แพ็ก 3 ชิ้น""","""7-Eleven"""


In [18]:
df_trans_7_eleven.write_excel(f"7_eleven_laundry_{today_date}.xlsx")